In [ ]:
"""
Consolidated Thesis Script for Implicit Media Bias Detection

This script implements the full pipeline:
1. Data loading
2. Preprocessing & cleaning
3. Exploratory Data Analysis (EDA)
4. Embeddings, UMAP & HDBSCAN clustering
5. Semantic drift calculation
6. Baseline classification (TF–IDF + Logistic Regression)
7. Supervised contrastive + classification modeling
8. Evaluation & results saving
"""

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from textblob import TextBlob
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import umap                  # pip install umap-learn
import hdbscan               # pip install hdbscan
from sentence_transformers import SentenceTransformer  # pip install sentence-transformers
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from transformers import AutoTokenizer, AutoModel         # pip install transformers

In [ ]:

# -----------------------------------------------------------------------------
# 1. Data Loading
# -----------------------------------------------------------------------------
RAW_PATH = 'data/raw/allsides_balanced_news_headlines-texts.csv'
PROCESSED_DIR = 'data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Loading raw data...")
raw_df = pd.read_csv(RAW_PATH)
print(f"  Total records: {len(raw_df)}")

In [ ]:

# -----------------------------------------------------------------------------
# 2. Preprocessing & Cleaning
# -----------------------------------------------------------------------------
def clean_text(text: str) -> str:
    text = re.sub(r'<[^>]+>', ' ', text)        # remove HTML tags
    text = re.sub(r'http\S+', ' ', text)        # remove URLs
    text = re.sub(r'[^A-Za-z0-9 ]+', ' ', text) # keep alphanumerics
    return text.lower().strip()

print("Cleaning text...")
raw_df['clean_text'] = raw_df['headline_text'].fillna("").apply(clean_text)
# remove very short entries
df = raw_df[raw_df['clean_text'].str.split().str.len() > 3].copy()
cleaned_path = os.path.join(PROCESSED_DIR, 'cleaned_news.csv')
df.to_csv(cleaned_path, index=False)
print(f"  Cleaned records: {len(df)}")
print(f"  Saved cleaned data to {cleaned_path}")

In [ ]:
# -----------------------------------------------------------------------------
# 3. Exploratory Data Analysis (EDA)
# -----------------------------------------------------------------------------
print("Running EDA...")
# Class balance bar chart
counts = df['bias_rating'].value_counts()
counts.plot(kind='bar', title="Class Balance")
plt.tight_layout()
plt.savefig('outputs/class_balance.png')
plt.close()

# Word clouds per class
for cls in counts.index:
    text = " ".join(df[df['bias_rating']==cls]['clean_text'])
    wc = WordCloud(width=800, height=400).generate(text)
    plt.figure(figsize=(8,4))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(f"Word Cloud: {cls}")
    plt.savefig(f'outputs/wordcloud_{cls}.png')
    plt.close()

# Sentiment polarity distribution
df['polarity'] = df['clean_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
plt.hist(df['polarity'], bins=30)
plt.title("Sentiment Polarity Distribution")
plt.tight_layout()
plt.savefig('outputs/sentiment_distribution.png')
plt.close()

# -----------------------------------------------------------------------------
# 4. Embeddings, UMAP & HDBSCAN Clustering
# -----------------------------------------------------------------------------
print("Generating embeddings...")
embed_model = SentenceTransformer('all-mpnet-base-v2')
embeddings = embed_model.encode(df['clean_text'].tolist(), show_progress_bar=True)
np.save(os.path.join(PROCESSED_DIR, 'embeddings.npy'), embeddings)

print("Reducing dimensionality with UMAP...")
umap_reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
umap_emb = umap_reducer.fit_transform(embeddings)
np.save(os.path.join(PROCESSED_DIR, 'umap_embeddings.npy'), umap_emb)

print("Clustering with HDBSCAN...")
clusterer = hdbscan.HDBSCAN(min_cluster_size=30)
clusters = clusterer.fit_predict(umap_emb)
df['cluster'] = clusters

plt.scatter(umap_emb[:,0], umap_emb[:,1], c=clusters, cmap='Spectral', s=5)
plt.title("UMAP + HDBSCAN Clusters")
plt.tight_layout()
plt.savefig('outputs/umap_hdbscan.png')
plt.close()

In [ ]:

# -----------------------------------------------------------------------------
# 5. Semantic Drift Calculation
# -----------------------------------------------------------------------------
print("Calculating semantic drift...")
centroids = {}
for cls in df['bias_rating'].unique():
    mask = df['bias_rating']==cls
    centroids[cls] = embeddings[mask].mean(axis=0)

def compute_drift(text, label):
    vec = embed_model.encode(text)
    return float(np.linalg.norm(vec - centroids[label]))

df['drift_score'] = df.apply(lambda row: compute_drift(row['clean_text'], row['bias_rating']), axis=1)
plt.hist(df['drift_score'], bins=50)
plt.title("Semantic Drift Distribution")
plt.tight_layout()
plt.savefig('outputs/drift_distribution.png')
plt.close()

In [ ]:

# -----------------------------------------------------------------------------
# 6. Baseline Classification (TF–IDF + Logistic Regression)
# -----------------------------------------------------------------------------
print("Training baseline logistic regression...")
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['clean_text'])
le = LabelEncoder()
y = le.fit_transform(df['bias_rating'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_test)

print("Baseline Logistic Regression Results:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:

# -----------------------------------------------------------------------------
# 7. Supervised Contrastive + Classification Modeling
# -----------------------------------------------------------------------------
print("Preparing supervised contrastive model...")
class BiasDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        txt = self.texts[idx]
        enc = self.tokenizer(
            txt,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

class ContrastiveClassifier(nn.Module):
    def __init__(self, model_name: str, num_labels: int):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden, num_labels)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        pooled = out.pooler_output
        features = nn.functional.normalize(pooled, p=2, dim=1)
        logits = self.classifier(self.dropout(pooled))
        return features, logits

# Data preparation
model_name = 'bucketresearch/politicalBiasBERT'
tokenizer = AutoTokenizer.from_pretrained(model_name)
texts = df['clean_text'].tolist()
labels = y.tolist()
X_tr, X_val, y_tr, y_val = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)
train_ds = BiasDataset(X_tr, y_tr, tokenizer)
val_ds   = BiasDataset(X_val, y_val, tokenizer)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32)

# Loss functions and optimizer
ce_loss = nn.CrossEntropyLoss()

class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature
        self.cos = nn.CosineSimilarity(dim=2)
    def forward(self, feats, labels):
        sim = torch.exp(self.cos(feats.unsqueeze(1), feats.unsqueeze(0)) / self.temperature)
        mask = torch.eq(labels.unsqueeze(1), labels.unsqueeze(0)).float().to(sim.device)
        pos = sim * mask
        loss = -torch.log(pos.sum(1) / (sim.sum(1) - pos.sum(1) + 1e-8)).mean()
        return loss

contrastive_loss_fn = NTXentLoss()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ContrastiveClassifier(model_name, num_labels=len(le.classes_)).to(device)
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

# Training loop
print("Training contrastive + classifier model...")
for epoch in range(3):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)
        feats, logits = model(input_ids, attention_mask)
        loss = ce_loss(logits, lbls) + contrastive_loss_fn(feats, lbls)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

# Evaluation
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)
        feats, logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(lbls.cpu().tolist())

print("Contrastive + Classification Results:")
print(classification_report(all_labels, all_preds, target_names=le.classes_))

In [ ]:
# -----------------------------------------------------------------------------
# 8. Save final artifacts
# -----------------------------------------------------------------------------
print("Saving artifacts...")
torch.save(model.state_dict(), 'models/contrastive_classifier.pt')
pd.DataFrame({'true': all_labels, 'pred': all_preds}).to_csv(
    os.path.join(PROCESSED_DIR, 'final_preds.csv'),
    index=False
)
print("Pipeline complete.")